In [ ]:
### New things:

# Import options:
# , nrows=n check the first n rows
# , usecols=["col1", "col2", etc] import only a subset of col  

# Exploration 
# .value_count()

# Assert
# Assert that two cols are equals, including the missings: assert col1.equals(col2)
# Assert that two cols have the same missings/non-missings: use .all() -> assert col1.all()==col2.all() 

In [ ]:
# Task 5: response rate from text to a number (target: 8 min)

# In the raw file, price is text such as "$1,234.00". Inside Airbnb uses $ even for prices in euros. 
# Your clean table already has price_eur, built in SQL, which gives you a perfect sanity check: your pandas conversion should give exactly the same numbers.

# Load id, price, bathrooms_text and amenities from the raw file. Show 5 price values and count the missing ones.
# Create price_num, a number, by removing the characters that aren’t digits or the decimal point, then converting. 
# Watch trap 2: $ is a special regex character.

import pandas as pd 
raw = pd.read_csv("../data/raw/listings.csv.gz", usecols=["id", "price", "bathrooms_text", "amenities"])

display(raw["price"].head(5))    # $ symbol
display(raw["price"].isna().sum())   # 33,612

raw["price_num"]=raw["price"].str.replace("$","", regex=False).str.replace(",","")
raw["price_num"]=pd.to_numeric(raw["price_num"], errors="coerce")
display(raw[["price", "price_num"]].sort_values("price_num", ascending=False).head(10))

raw["price_num_check"]=raw["price"].str.replace("$","").str.replace(",","")
raw["price_num_check"]=pd.to_numeric(raw["price_num_check"], errors="coerce")
assert raw["price_num_check"].equals(raw["price_num"])

# Sanity checks (as code):
# The number of missing values in price_num equals the number in price.
assert raw["price_num"].isna().sum() == raw["price"].isna().sum()
# Merge with listings_clean.parquet on id (use validate="1:1"), then check that price_num equals price_eur for every row. Hint: .equals() treats two NaNs in the same position as equal, whereas == doesn’t.
raw_clean = pd.read_parquet("../data/clean/listings_clean.parquet") 
join = pd.merge(
    raw,
    raw_clean,
    on="id",
    how="inner",
    validate="1:1"
)
assert len(join) == len(raw)
assert join["price_num"].equals(join["price_eur"]) #asser passes

display(raw["price_num"].describe())    # min 10, max 100,000: clearly outliers. The max moves the mean up, certainly better to look at median 

In [ ]:
### Task 6: number of bathrooms from text (target: 10 min)

# bathrooms_text looks like "1 bath", "1.5 baths", "2 shared baths" or "Half-bath". This task uses two new tools: extract and contains.

# The regex you need (it’s new, so here it is in full): r"(\d+\.?\d*)"
    # \d+: one or more digits
    # \.?: an optional decimal point
    # \d*: zero or more digits
    # ( ): the part to capture

# Show raw["bathrooms_text"].value_counts().head(15) to see the formats.
import pandas as pd 
raw = pd.read_csv("../data/raw/listings.csv.gz", usecols=["id", "price", "bathrooms_text", "room_type"])
display(raw["bathrooms_text"].value_counts().head(15))

# Create n_baths (a float) with .str.extract(..., expand=False), then convert it to a number.
# “Half-bath” contains no digit, so it gives NaN. Set those values to 0.5 using a contains("half", case=False, na=False) mask and .loc.
mask_1=raw["bathrooms_text"].str.lower().str.contains("half", case=False, na=False)
mask_2=raw["bathrooms_text"].str.lower().str.contains("no bathroom", case=False, na=False)  # See below sanity checks: "no bathroom" needs to be added 
raw["n_baths"]=raw["bathrooms_text"].str.extract(r"(\d+\.?\d*)", expand=False)
raw.loc[mask_1,"n_baths"]="0.5"
raw.loc[mask_2,"n_baths"]="0"
raw["n_baths"]=pd.to_numeric(raw["n_baths"], errors="coerce")

# Create is_shared: True if the text contains “shared” (any case).
raw["is_shared"] = raw["bathrooms_text"].str.lower().str.contains("shared")
display(raw[["bathrooms_text","n_baths","is_shared"]].sort_values("bathrooms_text", ascending=False).head(10))

# Sanity checks (as code):
    # No value that is non-missing in bathrooms_text should end up NaN in n_baths.
# assert raw["bathrooms_text"].notna().all()!=raw["n_baths"].isna().all()   
    # attempt but failed assert, yet I did not find any rows that have non-missing bathroom_text and missings n_baths.
    # I used the following :  display(raw
                            # .loc[ ((raw["bathrooms_text"].notna()) & (raw["n_baths"].isna())) ]
                            # .head(10)
                            # )
    # Using this, I found that "No bathroom" was not converted to a number so I added it above in mask_2
assert (raw["bathrooms_text"].isna()).equals(raw["n_baths"].isna()) # at least we make sure that all missings bathroom_text are also missings in n_baths
assert raw["bathrooms_text"].isna().sum()==raw["n_baths"].isna().sum()  # does not guarantee they are the same missing rows, but at least it is the same number
    # How many n_baths values differ from n_bathrooms in the clean table (after a merge on id)? Show 5 of them if any differ.
raw_clean = pd.read_parquet("../data/clean/listings_clean.parquet") 
join = pd.merge(
    raw, 
    raw_clean, 
    on = "id",
    how="inner",
    validate="1:1"
)
assert len(join)==len(raw)
assert join["n_baths"].equals(join["n_bathrooms"])

# 40min31 got stuck with the sanity check above 

# Expected output: the value counts, the mismatch count, and the share of shared bathrooms by room_type (4 rows).
# Share of shared bathrooms by room_type 
display(
    raw
    .groupby("room_type", as_index=False)["is_shared"]
    .mean()
    .round(4)
)

display(
    raw 
    .loc[
        (raw["room_type"]=="Shared room") & (raw["is_shared"]==False)
    ]
)

bathrooms_text
1 bath               55438
2 baths               5547
1.5 baths             5299
1 private bath        4032
1 shared bath         2556
2.5 baths             1395
3 baths                945
0 baths                413
Half-bath              256
1.5 shared baths       236
3.5 baths              223
4 baths                189
Shared half-bath       142
2 shared baths          91
Private half-bath       83
Name: count, dtype: int64

,bathrooms_text,n_baths,is_shared
2291,Shared half-bath,0.5,True
6210,Shared half-bath,0.5,True
11639,Shared half-bath,0.5,True
13020,Shared half-bath,0.5,True
14427,Shared half-bath,0.5,True
16310,Shared half-bath,0.5,True
18121,Shared half-bath,0.5,True
18620,Shared half-bath,0.5,True
18710,Shared half-bath,0.5,True
18898,Shared half-bath,0.5,True


,room_type,is_shared
0,Entire home/apt,0.0000
1,Hotel room,0.1788
2,Private room,0.3492
3,Shared room,0.9933


,id,room_type,bathrooms_text,price,n_baths,is_shared
28896,584713933358421577,Shared room,NaN,$60.00,NaN,False
